# RoPPT - VUA-18: resume from checkpoint, run final epoch + test eval (GPU)

1. **Upload** `roppt_upload.zip` to Google Drive root folder (My Drive) - drag & drop at drive.google.com
2. **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**
3. Authorize Google Drive when prompted
4. When finished, results are saved to My Drive as `roppt_results.zip`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/work
!unzip -q -o /content/drive/MyDrive/roppt_upload.zip -d /content/work
!ls -lh /content/work /content/work/saves/roberta-base/*/

In [ ]:
# Dependencies (GPU torch is preinstalled on Colab)
!pip install -q -U transformers nltk simplejson lxml tensorboardX
!python -c "import nltk; nltk.download('punkt', quiet=True)"
# Patch for newer transformers: AdamW moved to torch.optim (both importers)
!sed -i 's/from transformers import AutoTokenizer, AutoModel, AdamW, get_linear_schedule_with_warmup/from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup\ntry:\n    from transformers import AdamW\nexcept ImportError:\n    from torch.optim import AdamW/' /content/work/main.py
!sed -i 's/from transformers import AdamW, get_linear_schedule_with_warmup, get_cosine_with_hard_restarts_schedule_with_warmup/from transformers import get_linear_schedule_with_warmup, get_cosine_with_hard_restarts_schedule_with_warmup\ntry:\n    from transformers import AdamW\nexcept ImportError:\n    from torch.optim import AdamW/' /content/work/trainer.py
!grep -n "AdamW" /content/work/main.py /content/work/trainer.py | head -6
!cd /content/work && python -c 'import main; print("imports OK")'

In [ ]:
%cd /content/work
import glob, os
runs = sorted(glob.glob('saves/roberta-base/*/'), key=os.path.getmtime)
RUN_DIR = [r for r in runs if os.path.exists(os.path.join(r, 'pytorch_model.bin'))][-1].rstrip('/')
print('Resuming from:', RUN_DIR)
os.environ['RUN_DIR'] = RUN_DIR

In [ ]:
# Final epoch: resume from checkpoint, dev-select, then reload best and run full test eval.
# warmup_epoch 0 -> linear decay from 3e-5, mirroring the tail of the original 3-epoch schedule.
!python main.py --bert_model "$RUN_DIR" --data_dir data/VUA18 --task_name vua18 --dataset_name vua18 --model_type MELBERT_GAT --class_weight 3 --bert_model roberta-base --num_train_epoch 1 --warmup_epoch 0 --train_batch_size 16 --learning_rate 3e-5

In [ ]:
# Save results back to Google Drive
!zip -qr /content/drive/MyDrive/roppt_results.zip saves data/VUA18/error_idx_reshaped.txt
!ls -lh /content/drive/MyDrive/roppt_results.zip
print('DONE - download roppt_results.zip from My Drive')